# **Fama–French 5-Factor Model**
### **Factors:** `MKT` | `SMB` | `HML` | `RMW` | `CMA`

---

### **Overview**
- The **Fama–French 5-factor model** explains stock returns using **five systematic risk factors**
- It extends:
  - **CAPM**
  - **Fama–French 3-factor model**
- Adds:
  - **Profitability**
  - **Investment behavior**

---

### **Model Equation**
- **`Ri - Rf = α + βm(MKT) + βs(SMB) + βh(HML) + βr(RMW) + βc(CMA) + ε`**

---

### **Terms Explanation**
- **`Ri`** → Return of asset
- **`Rf`** → Risk-free rate
- **`MKT`** → Market return minus risk-free rate
- **`α` (Alpha)** → Excess return
- **`ε`** → Error term

---

### **Factor Definitions**

#### **1. `MKT` (Market Risk Premium)**
- **`MKT = Rm - Rf`**
- Captures overall market movement

---

#### **2. `SMB` (Small Minus Big)**
- Small-cap returns − Large-cap returns
- Proxy: Market capitalization

---

#### **3. `HML` (High Minus Low)**
- Value stocks − Growth stocks
- Proxy: Book-to-market ratio

---

#### **4. `RMW` (Robust Minus Weak)**
- High profitability − Low profitability
- Proxy: ROE, ROA

---

#### **5. `CMA` (Conservative Minus Aggressive)**
- Low investment − High investment
- Proxy: Asset growth

---

### **Intuition**
- Returns are driven by **systematic factor exposure**
- Each stock has **factor sensitivities (`β`)**
- These determine expected return

---

### **Why This Model Matters**
- Explains cross-sectional return differences
- Used in:
  - Portfolio construction
  - Quant investing
  - Asset pricing

---

### **Limitations**
- Linear assumptions
- Static factors
- Missing:
  - `Momentum`
  - Nonlinear effects

---

### **Project Connection**
- Construct all 5 factors from raw data
- Use them to:
  - Explain returns
  - Generate alpha
- Extend with ML

---

### **Expected Outputs**
- Factor series: `MKT`, `SMB`, `HML`, `RMW`, `CMA`
- Factor loadings
- Alpha
- Backtest results

### Phase 1: Data ingestion and Cleaning (Nifty 500: From 01-01-2011 till date)

In [1]:
# ==========================================
# Phase 1: Data Collection (NIFTY 100)
# ==========================================

from nselib import capital_market
import pandas as pd
from datetime import date, datetime
import time
import os
import requests
import feedparser
from urllib.parse import quote_plus
import shutil
from pathlib import Path
from transformers import pipeline


C:\Users\jayan\anaconda3\envs\ml_alpha_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# import sys
# !{sys.executable} -m pip install nselib
# !{sys.executable} -m pip install feedparser

In [3]:
# =====================================
# STEP 0: Project Setup
# =====================================

PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"

# Clean full data directory
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)

RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Clean data directories ready.")


Clean data directories ready.


In [4]:
FIN_PROCESSED_DIR = DATA_DIR / "fin_data_processed"
FIN_BACKUPS_DIR = DATA_DIR / "fin_data_backups"

FIN_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIN_BACKUPS_DIR.mkdir(parents=True, exist_ok=True)


In [5]:
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
# ====================================
# STEP 1: Fetch NIFTY 100 tickers
# ====================================

url = "https://archives.nseindia.com/content/indices/ind_nifty500list.csv"
df_list = pd.read_csv(url)

tickers = df_list["Symbol"].tolist()

print(f"Total tickers fetched: {len(tickers)}")


Total tickers fetched: 500


In [7]:
df_list.head()

,Company Name,Industry,Symbol,Series,ISIN Code
0,360 ONE WAM Ltd.,Financial Services,360ONE,EQ,INE466L01038
1,3M India Ltd.,Diversified,3MINDIA,EQ,INE470A01017
2,ABB India Ltd.,Capital Goods,ABB,EQ,INE117A01022
3,ACC Ltd.,Construction Materials,ACC,EQ,INE012A01025
4,ACME Solar Holdings Ltd.,Power,ACMESOLAR,EQ,INE622W01025


In [8]:
df_list.shape

(500, 5)

In [9]:
# ==================================
# STEP 2: Fetch Price + News
# ==================================

def fetch_all_data(symbols, start, end):

    failed = []

    for symbol in symbols:

        try:
            print(f"\n Fetching {symbol} ...")

            df = capital_market.price_volume_and_deliverable_position_data(
                symbol=symbol,
                from_date=start.strftime("%d-%m-%Y"),
                to_date=end.strftime("%d-%m-%Y"),
            )

            if df.empty:
                print(f" No price data for {symbol}")
                failed.append(symbol)
                continue

            df.columns = df.columns.str.lower().str.strip()

            # Keep only EQ series
            if "series" in df.columns:
                df = df[df["series"].str.upper() == "EQ"].copy()

                if df.empty:
                    print(f" {symbol} has no EQ data")
                    failed.append(symbol)
                    continue

            # Save price
            df.to_csv(RAW_DIR / f"{symbol}.csv", index=False)
            print(f" Price saved")

        except Exception as e:
            print(f"Failed: {symbol} | {e}")
            failed.append(symbol)

        time.sleep(1)

    print("\n Data Fetch Completed.")
    print(f" Success: {len(symbols) - len(failed)} | Failed: {len(failed)}")

    if failed:
        pd.DataFrame(failed, columns=["Symbol"]).to_csv(
            DATA_DIR / "failed_tickers.csv", index=False
        )
        print("Failed tickers saved.")

In [10]:
## =======================================================================
# STEP 3: Execute for retreiving stock price data of 500 stocks from 2011
# =======================================================================

START_DATE = date(2011, 1, 1)
END_DATE = date.today()

fetch_all_data(tickers, START_DATE, END_DATE)

print("Total RAW files:", len(list(RAW_DIR.glob("*.csv"))))


 Fetching 360ONE ...
 Price saved

 Fetching 3MINDIA ...
 Price saved

 Fetching ABB ...
 Price saved

 Fetching ACC ...
 Price saved

 Fetching ACMESOLAR ...
 Price saved

 Fetching AIAENG ...
 Price saved

 Fetching APLAPOLLO ...
 Price saved

 Fetching AUBANK ...
 Price saved

 Fetching AWL ...
 Price saved

 Fetching AADHARHFC ...
 Price saved

 Fetching AARTIIND ...
 Price saved

 Fetching AAVAS ...
 Price saved

 Fetching ABBOTINDIA ...
 Price saved

 Fetching ACE ...
 Price saved

 Fetching ACUTAAS ...
 Price saved

 Fetching ADANIENSOL ...
 Price saved

 Fetching ADANIENT ...
 Price saved

 Fetching ADANIGREEN ...
 Price saved

 Fetching ADANIPORTS ...
 Price saved

 Fetching ADANIPOWER ...
 Price saved

 Fetching ATGL ...
 Price saved

 Fetching ABCAPITAL ...
 Price saved

 Fetching ABFRL ...
 Price saved

 Fetching ABLBL ...
 Price saved

 Fetching ABREL ...
 Price saved

 Fetching ABSLAMC ...
 Price saved

 Fetching CPPLUS ...
 Price saved

 Fetching AEGISLOG ...
 Price sav


 Fetching IFCI ...
 Price saved

 Fetching IIFL ...
 Price saved

 Fetching IRB ...
 Price saved

 Fetching IRCON ...
 Price saved

 Fetching ITCHOTELS ...
 Price saved

 Fetching ITC ...
 Price saved

 Fetching ITI ...
 Price saved

 Fetching INDGN ...
 Price saved

 Fetching INDIACEM ...
 Price saved

 Fetching INDIAMART ...
 Price saved

 Fetching INDIANB ...
 Price saved

 Fetching IEX ...
 Price saved

 Fetching INDHOTEL ...
 Price saved

 Fetching IOC ...
 Price saved

 Fetching IOB ...
 Price saved

 Fetching IRCTC ...
 Price saved

 Fetching IRFC ...
 Price saved

 Fetching IREDA ...
 Price saved

 Fetching IGL ...
 Price saved

 Fetching INDUSTOWER ...
 Price saved

 Fetching INDUSINDBK ...
 Price saved

 Fetching NAUKRI ...
 Price saved

 Fetching INFY ...
 Price saved

 Fetching INOXWIND ...
 Price saved

 Fetching INTELLECT ...
 Price saved

 Fetching INDIGO ...
 Price saved

 Fetching IGIL ...
 Price saved

 Fetching IKS ...
 Price saved

 Fetching IPCALAB ...
 Price save

 Price saved

 Fetching RAMCOCEM ...
 Price saved

 Fetching THERMAX ...
 Price saved

 Fetching TIMKEN ...
 Price saved

 Fetching TITAGARH ...
 Price saved

 Fetching TITAN ...
 Price saved

 Fetching TORNTPHARM ...
 Price saved

 Fetching TORNTPOWER ...
 Price saved

 Fetching TARIL ...
 Price saved

 Fetching TRAVELFOOD ...
 Price saved

 Fetching TRENT ...
 Price saved

 Fetching TRIDENT ...
 Price saved

 Fetching TRITURBINE ...
 Price saved

 Fetching TIINDIA ...
 Price saved

 Fetching UCOBANK ...
 Price saved

 Fetching UNOMINDA ...
 Price saved

 Fetching UPL ...
 Price saved

 Fetching UTIAMC ...
 Price saved

 Fetching ULTRACEMCO ...
 Price saved

 Fetching UNIONBANK ...
 Price saved

 Fetching UBL ...
 Price saved

 Fetching UNITDSPR ...
 Price saved

 Fetching URBANCO ...
 Price saved

 Fetching USHAMART ...
 Price saved

 Fetching VTL ...
 Price saved

 Fetching VBL ...
 Price saved

 Fetching VEDL ...
 Price saved

 Fetching VIJAYA ...
 Price saved

 Fetching VMM ...
 P

In [11]:
# ======================================================
# STEP 4: Fetching Fundamental Data
# ======================================================

import warnings; warnings.filterwarnings("ignore")

folder_path = RAW_DIR
all_data = []

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        temp = pd.read_csv(os.path.join(folder_path, file), encoding='latin1')
        
        temp.columns = temp.columns.str.strip().str.lower()
        temp['symbol'] = file.replace(".csv", "")
        
        # Clean price
        temp['closeprice'] = temp['closeprice'].astype(str).str.replace(',', '')
        temp['closeprice'] = pd.to_numeric(temp['closeprice'], errors='coerce')
        
        # FIXED DATE PARSING
        temp['date'] = pd.to_datetime(temp['date'], dayfirst=True, errors='coerce')
        
        # Drop invalid dates
        temp = temp.dropna(subset=['date'])
        
        temp = temp.sort_values('date')
        temp.set_index('date', inplace=True)
        
        # Monthly resample
        monthly = temp.resample('M').last().reset_index()
        monthly['return'] = monthly['closeprice'].pct_change()
        
        monthly['year'] = monthly['date'].dt.year
        monthly['month'] = monthly['date'].dt.month
        
        monthly['effective_year'] = monthly['year']
        monthly.loc[monthly['month'] < 7, 'effective_year'] -= 1
        
        all_data.append(monthly)

price_data = pd.concat(all_data, ignore_index=True)
price_data = price_data.sort_values('date').reset_index(drop=True)

In [12]:
price_data.sort_values('date')

,date,"ã¯â»â¿""symbol""",series,prevclose,openprice,highprice,lowprice,lastprice,closeprice,averageprice,totaltradedquantity,turnoverinrs,no.oftrades,deliverableqty,%dlyqttotradedqty,symbol,return,year,month,effective_year
0,2011-01-31,UBL,EQ,443.65,435.55,484.0,425.2,482.4,474.00,458.36,110067.0,5.045075e+07,NaN,35788.0,32.51,UBL,NaN,2011,1,2010
181,2011-01-31,JPPOWER,EQ,45.75,45.65,46.45,44.0,45.3,45.10,44.89,340237.0,1.527195e+07,NaN,116466.0,34.23,JPPOWER,NaN,2011,1,2010
182,2011-01-31,GODREJCP,EQ,382.6,380.55,382.0,365.0,371.0,370.75,371.34,127345.0,4.728809e+07,NaN,103810.0,81.52,GODREJCP,NaN,2011,1,2010
183,2011-01-31,SAREGAMA,EQ,61.25,60.5,65.3,55.2,65.3,62.15,59.08,11404.0,6.737979e+05,NaN,5892.0,51.67,SAREGAMA,NaN,2011,1,2010
184,2011-01-31,HINDPETRO,EQ,365.2,361.1,364.0,350.1,358.85,359.10,357.81,1194764.0,4.275007e+08,NaN,549528.0,45.99,HINDPETRO,NaN,2011,1,2010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69122,2026-08-31,NBCC,EQ,96.25,96.01,96.44,94.78,95.33,95.33,95.45,4742088.0,4.526177e+08,21431.0,2011058.0,42.41,NBCC,0.005591,2026,8,2026
69121,2026-08-31,INDUSINDBK,EQ,"1,016.20","1,010.30","1,024.90","1,004.40","1,022.00",1022.00,"1,014.03",1727371.0,1.751607e+09,30202.0,985644.0,57.06,INDUSINDBK,0.009433,2026,8,2026
69120,2026-08-31,MSUMI,EQ,41.01,40.96,42.11,40.9,41.11,41.16,41.49,13810912.0,5.730801e+08,34893.0,6392965.0,46.29,MSUMI,0.013543,2026,8,2026
69133,2026-08-31,BLS,EQ,253.98,252.7,257.73,250.8,253.85,254.33,254.98,1226171.0,3.126510e+08,44150.0,596169.0,48.62,BLS,0.062364,2026,8,2026


In [13]:
# ==========================================================
# SAVE MONTHLY RETURNS
# ==========================================================

returns_df = price_data[
    ['date', 'symbol', 'return']
].copy()

returns_df.to_csv(
    PROCESSED_DIR / "monthly_returns.csv",
    index=False
)

print("Saved: monthly_returns.csv")

Saved: monthly_returns.csv


In [14]:
# Total number of monthly periods
total_periods = price_data['date'].nunique()

# Keep stocks with at least 70% data
threshold = 0.7 * total_periods

valid_stocks = (
    price_data.groupby('symbol')['closeprice']
    .count()
)

valid_stocks = valid_stocks[valid_stocks > threshold].index

price_data = price_data[price_data['symbol'].isin(valid_stocks)].copy()

In [15]:
median_vol = (
    price_data.groupby('symbol')['totaltradedquantity']
    .median()
)

liquid_stocks = median_vol[median_vol > 1e5].index

price_data = price_data[price_data['symbol'].isin(liquid_stocks)].copy()

In [16]:
price_data['totaltradedquantity'] = pd.to_numeric(
    price_data['totaltradedquantity'],
    errors='coerce'
)

In [17]:
price_data['turnover'] = (
    price_data['closeprice'] * price_data['totaltradedquantity']
)

In [18]:
# =====================================================================
# STEP 5: Fetching Risk-free returns (91-day Govt. Treasury bills)
# =====================================================================

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=INDIR3TIB01STM"

df = pd.read_csv(url)
df['DATE'] = pd.to_datetime(df['observation_date'])

df = df.set_index('DATE')
df.rename(columns={'INDIR3TIB01STM': 'Rf'}, inplace=True) #Rf means risk-free returns

df['Rf'] = df['Rf'] / 100
df = df.resample('D').ffill()
df['Rf_daily'] = df['Rf'] / 365

df = df[['Rf_daily']].copy()

df.to_csv(DATA_DIR / "rf_india.csv")

In [19]:
rf = pd.read_csv(DATA_DIR / "rf_india.csv", parse_dates=['DATE'])
rf = rf.set_index('DATE')

rf_monthly = rf.resample('M').apply(lambda x: (1 + x).prod() - 1)
rf_monthly = rf_monthly.rename(columns={'Rf_daily': 'Rf_monthly'})

In [20]:
rf_monthly.to_csv(
    PROCESSED_DIR / "rf_monthly.csv"
)

print("Saved: rf_monthly.csv")

Saved: rf_monthly.csv


In [21]:
# =====================================================================
# STEP 6: Financial Data Pipeline
# =====================================================================

In [22]:
import yfinance as yf
import requests
import pandas as pd
import numpy as np
import time, re, json
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def create_session():
    s = requests.Session()
    s.mount("https://", HTTPAdapter(max_retries=Retry(
        total=3, backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504])))
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "application/json, */*",
    })
    return s


In [23]:
symbols = tickers

In [24]:
# =============================================================================
# SOURCE 1 — yfinance with the correct API calls (not .quarterly_financials)
# Gets ~2015 onwards for most Nifty500 stocks
# =============================================================================

def fetch_yfinance(symbol):
    """
    Uses get_income_stmt / get_balance_sheet instead of the
    .quarterly_financials property — these hit a different Yahoo endpoint
    that returns more history (typically 12-16 quarters vs 4-5).
    """
    try:
        ticker = yf.Ticker(f"{symbol}.NS")

        # These methods return more history than the .quarterly_financials property
        inc = ticker.get_income_stmt(freq="quarterly", as_dict=False)
        bal = ticker.get_balance_sheet(freq="quarterly", as_dict=False)

        if inc is None or inc.empty:
            # Final fallback to the property
            inc = ticker.quarterly_financials
        if bal is None or bal.empty:
            bal = ticker.quarterly_balance_sheet

        if inc is None or inc.empty or bal is None or bal.empty:
            return None, "yf_empty"

        NI_KEYS = [
            "Net Income",
            "Net Income From Continuing Operations",
            "Net Income Including Noncontrolling Interests",
            "Net Income Common Stockholders",
        ]
        TA_KEYS = ["Total Assets"]

        ni_row = next((inc.loc[k] for k in NI_KEYS if k in inc.index), None)
        ta_row = next((bal.loc[k] for k in TA_KEYS if k in bal.index), None)

        if ni_row is None: return None, "yf_no_net_income"
        if ta_row is None: return None, "yf_no_assets"

        all_dates = sorted(
            set(ni_row.dropna().index) | set(ta_row.dropna().index)
        )

        records = []
        for dt in all_dates:
            month = dt.month
            if   month in [4,5,6]:   qtr, fy = "Q1", dt.year + 1
            elif month in [7,8,9]:   qtr, fy = "Q2", dt.year + 1
            elif month in [10,11,12]: qtr, fy = "Q3", dt.year + 1
            else:                     qtr, fy = "Q4", dt.year

            ni_val = ni_row.get(dt)
            ta_val = ta_row.get(dt)

            if pd.isna(ni_val) and pd.isna(ta_val):
                continue

            records.append({
                "year":         fy,
                "quarter":      qtr,
                "net_income":   float(ni_val) / 1e7 if pd.notna(ni_val) else None,
                "total_assets": float(ta_val) / 1e7 if pd.notna(ta_val) else None,
                "source":       "yfinance",
            })

        if not records:
            return None, "yf_no_records"

        df = (pd.DataFrame(records)
              .drop_duplicates(subset=["year", "quarter"])
              .sort_values(["year", "quarter"])
              .reset_index(drop=True))
        return df, None

    except Exception as e:
        return None, f"yf_error: {e}"


In [25]:
# =============================================================================
# SOURCE 2 — Screener.in annual data (HTML table scrape, NOT the JSON API)
# Gets 2014-2015 onwards reliably, annual only
# The HTML tables are far more stable than the JSON endpoint
# =============================================================================

from bs4 import BeautifulSoup

def fetch_screener_html(symbol, session):
    """
    Scrapes annual P&L and Balance Sheet from Screener.in HTML.
    
    KEY FIX: parse_table now keeps the FIRST non-empty occurrence of each
    row label. Screener's HTML contains duplicate labels (e.g. "Reserves"
    appears as a real data row, then again as an empty section-header row).
    The original dict-assignment silently overwrote real data with empty
    strings, causing book_value = None for every stock.
    """
 
    urls = [
        f"https://www.screener.in/company/{symbol}/consolidated/?view=10years",
        f"https://www.screener.in/company/{symbol}/?view=10years",
    ]
 
    # ------------------------------------------------------------------ #
    # FIXED parse_table: first non-empty occurrence of each label wins    #
    # ------------------------------------------------------------------ #
    def parse_table(tbl):
        rows = {}
        headers = []
        EMPTY_VALS = {'', '-', 'na', 'n/a', 'nan', '--'}
 
        for i, row in enumerate(tbl.find_all("tr")):
            cells = [c.get_text(strip=True) for c in row.find_all(["th", "td"])]
            if not cells:
                continue
 
            if i == 0:
                headers = cells
                continue
 
            label  = cells[0].strip()
            values = cells[1:]
 
            if not label:
                continue
 
            def row_has_data(vals):
                return any(
                    v.strip().lower() not in EMPTY_VALS for v in vals
                )
 
            if label not in rows:
                rows[label] = values
            else:
                # Only overwrite if stored value is all-empty AND new row has data.
                # First real data row always wins.
                if not row_has_data(rows[label]) and row_has_data(values):
                    rows[label] = values
 
        return headers, rows
 
    # ------------------------------------------------------------------ #
    # Safe value extractor                                                 #
    # ------------------------------------------------------------------ #
    def get_val(row_vals, col_idx):
        try:
            v = str(row_vals[col_idx]).replace(",", "").replace("(", "-").replace(")", "").strip()
            if v.lower() in {'', '-', 'na', 'n/a', 'nan', '--', 'ttm'}:
                return None
            return float(v)
        except (IndexError, ValueError, TypeError):
            return None
 
    for url in urls:
        try:
            res = session.get(url, timeout=15)
            if res.status_code != 200:
                continue
            if len(res.text) < 5000 or "No data" in res.text:
                continue
 
            soup = BeautifulSoup(res.text, "html.parser")
            sections = soup.find_all("section")
 
            pnl_table, bal_table = None, None
 
            for sec in sections:
                heading = sec.find("h2") or sec.find("h3")
                if not heading:
                    continue
                h = heading.text.lower().strip()
                table = sec.find("table")
                if not table:
                    continue
 
                if ("profit" in h or "income" in h or "p&l" in h) and pnl_table is None:
                    pnl_table = table
                elif "balance" in h and bal_table is None:
                    bal_table = table
 
            if pnl_table is None or bal_table is None:
                continue
 
            pnl_headers, pnl_rows = parse_table(pnl_table)
            bal_headers, bal_rows = parse_table(bal_table)
 
            # -------------------------------------------------------------- #
            # Year-to-column-index maps (built independently per table)       #
            # -------------------------------------------------------------- #
            def build_year_index(headers):
                result = []
                for idx, h in enumerate(headers[1:], start=0):
                    m = re.search(r'(20\d{2})', h)
                    if m:
                        result.append((idx, m.group(1)))
                return result
 
            pnl_year_indices = build_year_index(pnl_headers)
            bal_year_indices = build_year_index(bal_headers)
 
            if not pnl_year_indices:
                continue
 
            # Map: year_string -> col_index, for the balance sheet
            bal_year_map = {yr: idx for idx, yr in bal_year_indices}
 
            # -------------------------------------------------------------- #
            # Locate net income row (P&L)                                     #
            # -------------------------------------------------------------- #
            NI_RE = re.compile(r"net profit|profit after tax|\bpat\b", re.I)
            ni_vals_raw = None
            for label, vals in pnl_rows.items():
                if NI_RE.search(label):
                    ni_vals_raw = vals
                    break
 
            # -------------------------------------------------------------- #
            # Locate total assets row (Balance Sheet)                         #
            # -------------------------------------------------------------- #
            ta_vals_raw = None
            for label, vals in bal_rows.items():
                if "total assets" in label.lower().strip():
                    ta_vals_raw = vals
                    break
 
            # -------------------------------------------------------------- #
            # Locate book value components (Balance Sheet)                    #
            #                                                                 #
            # Strategy (in priority order):                                   #
            #   1. Direct equity row: "Total Equity", "Shareholders' Equity", #
            #      "Net Worth", "Total Stockholders Equity" (IND-AS single)   #
            #   2. Computed: Share Capital + Reserves                         #
            # -------------------------------------------------------------- #
 
            DIRECT_EQUITY_RE = re.compile(
                r"total equity|shareholders['\s]*equity|net worth|"
                r"stockholders['\s]*equity|total networth|equity \(net worth\)",
                re.I
            )
 
            SC_RE = re.compile(
                r"equity capital|equity share capital|share capital",
                re.I
            )
            RES_RE = re.compile(
                r"\breserves\b|other equity|surplus",
                re.I
            )
 
            direct_equity_raw = None
            share_capital_raw = None
            reserve_raw       = None
 
            for label, vals in bal_rows.items():
                if direct_equity_raw is None and DIRECT_EQUITY_RE.search(label):
                    direct_equity_raw = vals
                if share_capital_raw is None and SC_RE.search(label):
                    share_capital_raw = vals
                if reserve_raw is None and RES_RE.search(label):
                    reserve_raw = vals
 
            # -------------------------------------------------------------- #
            # Build records                                                    #
            # -------------------------------------------------------------- #
            records = []
            for pnl_col_idx, yr in pnl_year_indices:
 
                # P&L column uses pnl_col_idx directly
                ni = get_val(ni_vals_raw, pnl_col_idx) if ni_vals_raw is not None else None
 
                # Balance sheet column: look up by year, fall back to same index
                bal_col_idx = bal_year_map.get(yr, pnl_col_idx)
 
                ta = get_val(ta_vals_raw, bal_col_idx) if ta_vals_raw is not None else None
 
                # Book value: try direct equity first, then computed
                bv = None
                if direct_equity_raw is not None:
                    bv = get_val(direct_equity_raw, bal_col_idx)
 
                if bv is None and share_capital_raw is not None and reserve_raw is not None:
                    sc = get_val(share_capital_raw, bal_col_idx)
                    rv = get_val(reserve_raw, bal_col_idx)
                    if sc is not None and rv is not None:
                        bv = sc + rv
 
                records.append({
                    "year":         int(yr),
                    "quarter":      "FY",
                    "net_income":   ni,
                    "total_assets": ta,
                    "book_value":   bv,
                    "source":       "screener_html",
                })
 
            if not records:
                continue
 
            df = (
                pd.DataFrame(records)
                .drop_duplicates("year")
                .sort_values("year")
                .reset_index(drop=True)
            )
            return df, None
 
        except Exception as e:
            print(f"[Screener ERROR] {symbol}: {e}")
            continue
 
        time.sleep(1.5)
 
    return None, "screener_html_failed"

In [26]:
# =============================================================================
# SOURCE 3 — NSE India quarterly results (XBRL XML, publicly accessible)
# More reliable than BSE for unauthenticated requests
# Gets 2014 onwards quarterly for most stocks
# =============================================================================

def fetch_nse_quarterly(symbol, session):
    """
    NSE publishes quarterly result XMLs via their public archive.
    No authentication required. Endpoint is stable.
    """
    url = (
        f"https://www.nseindia.com/api/results-comparisons?"
        f"index=equities&symbol={symbol}"
    )
    try:
        # NSE requires these specific headers
        nse_session = requests.Session()
        nse_session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
            "Accept": "application/json",
            "Referer": "https://www.nseindia.com/",
            "Accept-Language": "en-US,en;q=0.9",
        })
        # NSE requires a cookie — get it first with a homepage visit
        nse_session.get("https://www.nseindia.com", timeout=10)
        time.sleep(1)

        res = nse_session.get(url, timeout=15)
        if res.status_code != 200:
            return None, f"nse_status_{res.status_code}"

        data = res.json()
        rows = data.get("data", [])
        if not rows:
            return None, "nse_no_data"

        records = []
        for row in rows:
            period = row.get("period", "")          # e.g. "Q2 FY2023"
            ni  = row.get("net_profit_loss")
            ta  = row.get("total_assets")

            qm = re.search(r'(Q[1-4])\s*FY(\d{2,4})', period, re.I)
            if qm:
                qtr = qm.group(1).upper()
                fy  = int(qm.group(2))
                fy  = 2000 + fy if fy < 100 else fy
            else:
                continue

            try:
                records.append({
                    "year":         fy,
                    "quarter":      qtr,
                    "net_income":   float(str(ni).replace(",","")) if ni else None,
                    "total_assets": float(str(ta).replace(",","")) if ta else None,
                    "source":       "nse_api",
                })
            except (ValueError, TypeError):
                continue

        if not records:
            return None, "nse_no_parseable_rows"

        df = (pd.DataFrame(records)
              .drop_duplicates(subset=["year","quarter"])
              .sort_values(["year","quarter"])
              .reset_index(drop=True))
        return df, None

    except Exception as e:
        return None, f"nse_error: {e}"


In [27]:
# =============================================================================
# MASTER PIPELINE
# =============================================================================

def run_pipeline(symbol, session):
    results = []

    # Source 1: yfinance (~2015 onwards, quarterly)
    df, err = fetch_yfinance(symbol)
    if df is not None and len(df) > 0:
        df.insert(0, "symbol", symbol)
        results.append(df)
        print(f"  yf: {len(df)} qtrs ({df['year'].min()}-{df['year'].max()})", end="")

    # Source 2: Screener HTML (~2014 onwards, annual)
    df, err = fetch_screener_html(symbol, session)
    if df is not None and len(df) > 0:
        df.insert(0, "symbol", symbol)
        results.append(df)
        print(f"  scr: {len(df)} yrs", end="")

    # Source 3: NSE API (~2014 onwards, quarterly)
    df, err = fetch_nse_quarterly(symbol, session)
    if df is not None and len(df) > 0:
        df.insert(0, "symbol", symbol)
        results.append(df)
        print(f"  nse: {len(df)} qtrs", end="")

    if not results:
        return None, "all_sources_failed"

    # Merge all sources — quarterly beats annual for same period
    # Priority: yfinance > nse_api > screener_html
    combined = pd.concat(results, ignore_index=True)
    
    # --- Extract book_value separately BEFORE deduplication ---
    # Screener carries book_value; yfinance/NSE do not.
    # Deduplication (yfinance wins) would discard Screener rows and lose book_value.
    bv_df = (
        combined[combined["book_value"].notna()][["symbol", "year", "quarter", "book_value"]]
        .drop_duplicates(subset=["symbol", "year"])
        .copy()
    ) if "book_value" in combined.columns else pd.DataFrame()

    # Deduplicate financial rows by source priority (yfinance > nse_api > screener_html)
    priority = {"yfinance": 0, "nse_api": 1, "screener_html": 2}
    combined["_rank"] = combined["source"].map(priority).fillna(9)
    combined = (combined
        .sort_values("_rank")
        .drop_duplicates(subset=["symbol", "year", "quarter"], keep="first")
        .drop(columns="_rank")
        .sort_values(["year", "quarter"])
        .reset_index(drop=True))

    # --- Re-attach book_value after dedup ---
    if not bv_df.empty:
        if "book_value" in combined.columns:
            combined = combined.drop(columns=["book_value"])
        combined = combined.merge(
            bv_df[["symbol", "year", "book_value"]].drop_duplicates(subset=["symbol", "year"]),
            on=["symbol", "year"],
            how="left"
        )

    return combined, None

In [28]:
# =============================================================================
# EXECUTION
# =============================================================================

session = create_session()
all_data, failures = [], {}

for i, sym in enumerate(symbols):
    print(f"\n[{i+1}/{len(symbols)}] {sym}", end="  ")

    df, err = run_pipeline(sym, session)

    if df is not None:
        df["net_income"]   = pd.to_numeric(df["net_income"],   errors="coerce")
        df["total_assets"] = pd.to_numeric(df["total_assets"], errors="coerce")
        df = df[df["total_assets"] > 0].dropna(subset=["net_income"])

        if len(df) > 0:
            all_data.append(df)
            print(f"  → {len(df)} total rows")
        else:
            failures[sym] = "all_rows_invalid"
            print(f"  → all rows invalid after cleaning")
    else:
        failures[sym] = err
        print(f"  → FAILED: {err}")

    time.sleep(1.5 if (i+1) % 20 != 0 else 5.0)

    if (i+1) % 50 == 0 and all_data:
        pd.concat(all_data, ignore_index=True).to_csv(FIN_BACKUPS_DIR / f"fundamentals_backup_{i+1}.csv")
        print(f"\n  [backup saved at step {i+1}]")


final_df = pd.concat(all_data, ignore_index=True)

# Debug: confirm book_value presence
print("Columns in final_df:", final_df.columns.tolist())
print("book_value non-null count:", final_df['book_value'].notna().sum() if 'book_value' in final_df.columns else "COLUMN MISSING")
screener_rows = final_df[final_df['source'] == 'screener_html'] if 'source' in final_df.columns else pd.DataFrame()
print("Screener rows:", len(screener_rows))
if not screener_rows.empty and 'book_value' in screener_rows.columns:
    print("Screener book_value non-null:", screener_rows['book_value'].notna().sum())

if "book_value" not in final_df.columns:
    final_df["book_value"] = np.nan

# ==========================================================
# CLEAN BOOK VALUE PIPELINE
# ==========================================================

# Convert safely
final_df['book_value'] = pd.to_numeric(
    final_df['book_value'],
    errors='coerce'
)

# Keep only required columns
book_df = final_df[
    ['symbol', 'year', 'quarter', 'book_value']
].copy()

# After book_df is built, verify year alignment
print("book_df year sample:\n", book_df[['symbol','year','quarter','book_value']].head(10))

# Remove missing values
book_df = book_df.dropna(subset=['book_value'])

print("Initial non-null BV rows:", len(book_df))

# Keep annual values
book_df = book_df[
    book_df['quarter'].isin(['FY', 'Q4', 'Q3', 'Q2', 'Q1'])
]

print("Annual BV rows:", len(book_df))

# Create fiscal year-end date
book_df['date'] = pd.to_datetime(
    book_df['year'].astype(str) + "-03-31"
)

# Remove duplicates
book_df = (
    book_df
    .sort_values(['symbol', 'date'])
    .drop_duplicates(['symbol', 'date'], keep='last')
)

# Convert Cr → ₹
book_df['book_value'] = book_df['book_value'] * 1e7

# ==========================================================
# Monthly expansion
# ==========================================================

monthly_frames = []

for sym in book_df['symbol'].unique():

    temp = (
        book_df[book_df['symbol'] == sym]
        .sort_values('date')
        .set_index('date')[['book_value']]
        .resample('M')
        .ffill()
        .reset_index()
    )

    temp['symbol'] = sym

    monthly_frames.append(temp)

# Safe concat
if len(monthly_frames) > 0:

    book_monthly = pd.concat(
        monthly_frames,
        ignore_index=True
    )

else:
    print("WARNING: book_monthly is empty. Diagnostics:")
    print(f"  book_df shape: {book_df.shape}")
    print(f"  final_df book_value non-null: {final_df['book_value'].notna().sum()}")
    raise ValueError(
        "book_monthly is empty — check that Screener is returning book_value "
        "and that deduplication is not dropping all Screener rows."
    )
    
# Lag to avoid lookahead bias
book_monthly['book_value'] = (
    book_monthly
    .groupby('symbol')['book_value']
    .shift(3)
)

print(
    "Book monthly non-null:",
    book_monthly['book_value'].notna().sum()
)

final_df = final_df[final_df["year"].astype(int) >= 2011].reset_index(drop=True)
final_df = final_df.sort_values(["symbol","year","quarter"]).reset_index(drop=True)

pd.DataFrame(list(failures.items()), columns=["symbol","reason"]).to_csv(FIN_BACKUPS_DIR / "failures.csv", index=False)

print(f"\nSymbols     : {final_df['symbol'].nunique()}")
print(f"Total rows  : {len(final_df)}")
print(f"Year range  : {final_df['year'].min()} → {final_df['year'].max()}")
print(f"Avg rows/co : {len(final_df)/final_df['symbol'].nunique():.1f}")
print(f"Failures    : {len(failures)}")
print(f"\nSource breakdown:\n{final_df['source'].value_counts()}")


[1/500] 360ONE    scr: 9 yrs  → 9 total rows

[2/500] 3MINDIA    scr: 7 yrs  → 7 total rows

[3/500] ABB    scr: 4 yrs  → 4 total rows

[4/500] ACC    scr: 12 yrs  → 12 total rows

[5/500] ACMESOLAR    scr: 7 yrs  → 7 total rows

[6/500] AIAENG    scr: 12 yrs  → 12 total rows

[7/500] APLAPOLLO    scr: 12 yrs  → 12 total rows

[8/500] AUBANK    scr: 4 yrs  → 4 total rows

[9/500] AWL    scr: 8 yrs  → 8 total rows

[10/500] AADHARHFC    scr: 8 yrs  → 8 total rows

[11/500] AARTIIND    scr: 12 yrs  → 12 total rows

[12/500] AAVAS    scr: 7 yrs  → 7 total rows

[13/500] ABBOTINDIA    scr: 12 yrs  → 12 total rows

[14/500] ACE    scr: 12 yrs  → 12 total rows

[15/500] ACUTAAS    scr: 10 yrs  → 10 total rows

[16/500] ADANIENSOL    scr: 12 yrs  → 12 total rows

[17/500] ADANIENT    scr: 12 yrs  → 12 total rows

[18/500] ADANIGREEN    scr: 11 yrs  → 11 total rows

[19/500] ADANIPORTS    scr: 12 yrs  → 12 total rows

[20/500] ADANIPOWER    scr: 12 yrs  → 12 total rows

[21/500] ATGL    scr: 

[162/500] ETERNAL    scr: 9 yrs  → 9 total rows

[163/500] EXIDEIND    scr: 12 yrs  → 12 total rows

[164/500] NYKAA    scr: 9 yrs  → 9 total rows

[165/500] FEDERALBNK    scr: 12 yrs  → 12 total rows

[166/500] FACT    scr: 11 yrs  → 11 total rows

[167/500] FINCABLES    scr: 11 yrs  → 11 total rows

[168/500] FSL    scr: 12 yrs  → 12 total rows

[169/500] FIVESTAR    scr: 5 yrs  → 5 total rows

[170/500] FORCEMOT    scr: 12 yrs  → 12 total rows

[171/500] FORTIS    scr: 12 yrs  → 12 total rows

[172/500] GAIL    scr: 12 yrs  → 12 total rows

[173/500] GVT&D    scr: 1 yrs  → 1 total rows

[174/500] GMRAIRPORT    scr: 12 yrs  → 12 total rows

[175/500] GABRIEL    scr: 3 yrs  → 3 total rows

[176/500] GALLANTT    scr: 12 yrs  → 12 total rows

[177/500] GRSE    scr: 12 yrs  → 12 total rows

[178/500] GICRE    scr: 12 yrs  → 12 total rows

[179/500] GILLETTE    scr: 12 yrs  → 12 total rows

[180/500] GLAND    scr: 7 yrs  → 7 total rows

[181/500] GLAXO    scr: 12 yrs  → 12 total rows

[18

[323/500] MOTILALOFS    scr: 12 yrs  → 12 total rows

[324/500] MPHASIS    scr: 12 yrs  → 12 total rows

[325/500] MCX    scr: 12 yrs  → 12 total rows

[326/500] MUTHOOTFIN    scr: 12 yrs  → 12 total rows

[327/500] NATCOPHARM    scr: 12 yrs  → 12 total rows

[328/500] NBCC    scr: 12 yrs  → 12 total rows

[329/500] NCC    scr: 12 yrs  → 12 total rows

[330/500] NHPC    scr: 12 yrs  → 12 total rows

[331/500] NLCINDIA    scr: 12 yrs  → 12 total rows

[332/500] NMDC    scr: 12 yrs  → 12 total rows

[333/500] NSLNISP    scr: 7 yrs  → 5 total rows

[334/500] NTPCGREEN    scr: 4 yrs  → 4 total rows

[335/500] NTPC    scr: 12 yrs  → 12 total rows

[336/500] NH    scr: 12 yrs  → 12 total rows

[337/500] NATIONALUM    scr: 12 yrs  → 12 total rows

[338/500] NAVA    scr: 12 yrs  → 12 total rows

[339/500] NAVINFLUOR    scr: 12 yrs  → 12 total rows

[340/500] NESTLEIND    scr: 4 yrs  → 3 total rows

[341/500] NETWEB    scr: 8 yrs  → 8 total rows

[342/500] NEULANDLAB    scr: 12 yrs  → 12 total 


[484/500] VMM    scr: 7 yrs  → 7 total rows

[485/500] IDEA    scr: 12 yrs  → 12 total rows

[486/500] VOLTAS    scr: 12 yrs  → 12 total rows

[487/500] WAAREEENER    scr: 7 yrs  → 7 total rows

[488/500] WELCORP    scr: 12 yrs  → 12 total rows

[489/500] WELSPUNLIV    scr: 12 yrs  → 12 total rows

[490/500] WHIRLPOOL    scr: 9 yrs  → 9 total rows

[491/500] WIPRO    scr: 12 yrs  → 12 total rows

[492/500] WOCKPHARMA    scr: 12 yrs  → 12 total rows

[493/500] YESBANK    scr: 12 yrs  → 12 total rows

[494/500] ZFCVINDIA    scr: 5 yrs  → 5 total rows

[495/500] ZEEL    scr: 12 yrs  → 12 total rows

[496/500] ZENTEC    scr: 12 yrs  → 12 total rows

[497/500] ZENSARTECH    scr: 12 yrs  → 12 total rows

[498/500] ZYDUSLIFE    scr: 12 yrs  → 12 total rows

[499/500] ZYDUSWELL    scr: 12 yrs  → 12 total rows

[500/500] ECLERX    scr: 12 yrs  → 12 total rows

  [backup saved at step 500]
Columns in final_df: ['symbol', 'year', 'quarter', 'net_income', 'total_assets', 'source', 'book_value']
b

In [89]:
# Got errors several times while fetching the data, so running diagnostics for a single company to check the tags

"""
Run this as a STANDALONE cell in your notebook BEFORE the main pipeline.
It will print the exact HTML structure Screener returns for one stock,
so we can see exactly what parse_table is working with.
"""

import requests
from bs4 import BeautifulSoup
import re

def diagnose_screener(symbol="RELIANCE"):
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml",
        "Accept-Language": "en-US,en;q=0.9",
    })

    urls = [
        f"https://www.screener.in/company/{symbol}/consolidated/?view=10years",
        f"https://www.screener.in/company/{symbol}/?view=10years",
    ]

    for url in urls:
        print(f"\nFetching: {url}")
        res = session.get(url, timeout=15)
        print(f"Status: {res.status_code}  |  Length: {len(res.text)}")
        if res.status_code != 200:
            continue

        soup = BeautifulSoup(res.text, "html.parser")

        # ── Print ALL section headings ──────────────────────────────────────
        print("\n=== ALL SECTIONS ===")
        for sec in soup.find_all("section"):
            h = sec.find("h2") or sec.find("h3")
            has_table = sec.find("table") is not None
            print(f"  heading: '{h.text.strip() if h else None}'  |  has_table: {has_table}")

        # ── Find balance sheet table ────────────────────────────────────────
        bal_table = None
        for sec in soup.find_all("section"):
            h = sec.find("h2") or sec.find("h3")
            if h and "balance" in h.text.lower():
                bal_table = sec.find("table")
                break

        if bal_table is None:
            print("\n!!! NO BALANCE SHEET TABLE FOUND !!!")
            continue

        # ── Print every raw row of the balance sheet table ─────────────────
        print("\n=== BALANCE SHEET TABLE — RAW ROWS ===")
        for i, row in enumerate(bal_table.find_all("tr")):
            cells = [c.get_text(strip=True) for c in row.find_all(["th", "td"])]
            print(f"  Row {i:02d}: {cells}")

        # ── Show what parse_table (old) would produce ───────────────────────
        print("\n=== parse_table RESULT (label → first 3 values) ===")
        rows_dict = {}
        headers = []
        for i, row in enumerate(bal_table.find_all("tr")):
            cells = [c.get_text(strip=True) for c in row.find_all(["th", "td"])]
            if not cells:
                continue
            if i == 0:
                headers = cells
                print(f"  HEADERS: {headers}")
                continue
            rows_dict[cells[0]] = cells[1:]

        for label, vals in rows_dict.items():
            print(f"  '{label}': {vals[:3]}")

        # ── Specifically look for equity-related rows ───────────────────────
        print("\n=== EQUITY-RELATED ROW SEARCH ===")
        equity_patterns = [
            "share capital", "equity share", "reserves", "other equity",
            "surplus", "net worth", "total equity", "shareholders",
            "stockholders", "networth"
        ]
        for label, vals in rows_dict.items():
            lc = label.lower()
            if any(p in lc for p in equity_patterns):
                print(f"  MATCH '{label}': {vals[:5]}")

        print("\n=== DONE ===")
        return  # stop after first successful URL

diagnose_screener("RELIANCE")


Fetching: https://www.screener.in/company/RELIANCE/consolidated/?view=10years
Status: 200  |  Length: 227537

=== ALL SECTIONS ===
  heading: 'None'  |  has_table: False
  heading: 'None'  |  has_table: False
  heading: 'Peer comparison'  |  has_table: False
  heading: 'Quarterly Results'  |  has_table: True
  heading: 'Profit & Loss'  |  has_table: True
  heading: 'Balance Sheet'  |  has_table: True
  heading: 'Cash Flows'  |  has_table: True
  heading: 'Ratios'  |  has_table: True
  heading: 'Insights'  |  has_table: True
  heading: 'Shareholding Pattern'  |  has_table: True
  heading: 'Documents'  |  has_table: False

=== BALANCE SHEET TABLE — RAW ROWS ===
  Row 00: ['', 'Mar 2015', 'Mar 2016', 'Mar 2017', 'Mar 2018', 'Mar 2019', 'Mar 2020', 'Mar 2021', 'Mar 2022', 'Mar 2023', 'Mar 2024', 'Mar 2025', 'Mar 2026']
  Row 01: ['Equity Capital', '2,943', '2,948', '2,959', '5,922', '5,926', '6,339', '6,445', '6,765', '6,766', '6,766', '13,532', '13,532']
  Row 02: ['Reserves', '215,556',

In [90]:
def get_date(row):
    q_map = {'Q1': '06-30', 'Q2': '09-30', 'Q3': '12-31', 'Q4': '03-31', 'FY': '03-31'}
    return pd.to_datetime(f"{row['year']}-{q_map[row['quarter']]}")

final_df['date'] = final_df.apply(get_date, axis=1)

In [91]:
final_df.head()

,symbol,year,quarter,net_income,total_assets,source,book_value,date
date,,,,,,,,
2018-03-31,360ONE,2018,FY,380.0,9567.0,screener_html,1863.0,2018-03-31
2019-03-31,360ONE,2019,FY,375.0,9763.0,screener_html,2911.0,2019-03-31
2020-03-31,360ONE,2020,FY,201.0,13021.0,screener_html,2991.0,2020-03-31
2021-03-31,360ONE,2021,FY,369.0,8739.0,screener_html,2828.0,2021-03-31
2022-03-31,360ONE,2022,FY,578.0,10734.0,screener_html,3024.0,2022-03-31


In [92]:
final_df['date'] = pd.to_datetime(final_df['date'])

In [96]:
final_df.reset_index(drop=True, inplace=True)

In [97]:
final_df = final_df.sort_values(['symbol', 'date'])

In [98]:
final_df = final_df.set_index('date')

In [99]:
fund_monthly = (
    final_df
    .groupby('symbol', group_keys=False)
    .resample('M')
    .ffill()
    .reset_index()
)

In [100]:
fund_monthly.head()

,date,symbol,year,quarter,net_income,total_assets,source,book_value
0,2018-03-31,360ONE,2018,FY,380.0,9567.0,screener_html,1863.0
1,2018-04-30,360ONE,2018,FY,380.0,9567.0,screener_html,1863.0
2,2018-05-31,360ONE,2018,FY,380.0,9567.0,screener_html,1863.0
3,2018-06-30,360ONE,2018,FY,380.0,9567.0,screener_html,1863.0
4,2018-07-31,360ONE,2018,FY,380.0,9567.0,screener_html,1863.0


In [101]:
cols_to_lag = ['net_income', 'total_assets', 'book_value']

fund_monthly[cols_to_lag] = (
    fund_monthly.groupby('symbol')[cols_to_lag]
    .shift(3)
)

In [102]:
# import sys
# !{sys.executable} -m pip install plotly

In [103]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'browser'   # safest across environments

# Step 1: Count + sort years
year_counts = (
    final_df['year']
    .astype(int)
    .value_counts()
    .sort_index()
    .reset_index()
)

year_counts.columns = ['year', 'count']

# Step 2: Plot
fig = px.bar(
    year_counts,
    x='year',
    y='count',
    title='Number of Data Points per Year',
    labels={'year': 'Year', 'count': 'Data Availability: Number of stocks out of 500'}
)

# Optional styling
fig.update_layout(
    xaxis=dict(dtick=1),   # show every year
    template='plotly_white'
)

fig.show()

In [104]:
final_df.shape

(5143, 7)

In [105]:
final_df.head()

,symbol,year,quarter,net_income,total_assets,source,book_value
date,,,,,,,
2018-03-31,360ONE,2018,FY,380.0,9567.0,screener_html,1863.0
2019-03-31,360ONE,2019,FY,375.0,9763.0,screener_html,2911.0
2020-03-31,360ONE,2020,FY,201.0,13021.0,screener_html,2991.0
2021-03-31,360ONE,2021,FY,369.0,8739.0,screener_html,2828.0
2022-03-31,360ONE,2022,FY,578.0,10734.0,screener_html,3024.0


In [106]:
final_df.to_csv(FIN_PROCESSED_DIR / "final_fin_data.csv")

In [107]:
price_data.head()

,date,"symbol""",series,prevclose,openprice,highprice,lowprice,lastprice,closeprice,averageprice,...,turnoverinrs,no.oftrades,deliverableqty,%dlyqttotradedqty,symbol,return,year,month,effective_year,turnover
0,2011-01-31,UBL,EQ,443.65,435.55,484.0,425.2,482.4,474.00,458.36,...,5.045075e+07,NaN,35788.0,32.51,UBL,NaN,2011,1,2010,5.217176e+07
1,2011-01-31,BRITANNIA,EQ,368.35,371.9,382.0,365.0,380.95,378.30,375.88,...,1.538416e+07,NaN,26049.0,63.65,BRITANNIA,NaN,2011,1,2010,1.548306e+07
2,2011-01-31,MAGMA,EQ,60.65,59.9,60.85,58.0,60.0,60.05,59.74,...,1.198870e+07,NaN,108926.0,54.28,POONAWALLA,NaN,2011,1,2010,1.205041e+07
3,2011-01-31,GABRIEL,EQ,45.25,45.0,45.0,41.8,42.6,42.70,42.65,...,1.542479e+06,NaN,17264.0,47.74,GABRIEL,NaN,2011,1,2010,1.544288e+06
4,2011-01-31,PFC,EQ,254.45,247.2,251.0,240.55,245.85,245.20,245.36,...,1.791040e+08,NaN,487553.0,66.79,PFC,NaN,2011,1,2010,1.789855e+08


In [108]:
price_data.columns = price_data.columns.str.replace('ã¯â»â¿"', '', regex=False)
price_data.columns = price_data.columns.str.strip()

In [109]:
# =====================================================================
# STEP 7: Derive Factor Inputs
# =====================================================================

In [110]:
# =====================================================================
# 7.1 Profitability
# =====================================================================

In [111]:
#profitability or ROE
fund_monthly['profitability'] = (
    fund_monthly['net_income'] / fund_monthly['book_value']
)

In [112]:
# =====================================================================
# 7.2 CMA (Investment)
# =====================================================================

In [113]:
fund_annual = (
    fund_monthly
    .sort_values(['symbol', 'year'])
    .drop_duplicates(['symbol', 'year'], keep='last')
    .copy()
)

fund_annual['asset_growth'] = (
    fund_annual.groupby('symbol')['total_assets']
    .pct_change()
)

In [114]:
rf_monthly.head()

,date,Rf_monthly
0,2011-11-30,0.007315
1,2011-12-31,0.007290
2,2012-01-31,0.007342
3,2012-02-29,0.007110
4,2012-03-31,0.007692


In [115]:
rf_monthly = rf_monthly.reset_index()

In [116]:
rf_monthly.rename(columns={'DATE': 'date'}, inplace=True)

In [117]:
price_data['date'] = price_data['date'] + pd.offsets.MonthEnd(0)
fund_monthly['date'] = fund_monthly['date'] + pd.offsets.MonthEnd(0)

In [118]:
final = price_data.merge(fund_monthly, on=['symbol', 'date'], how='left')
final = final.merge(rf_monthly, on='date', how='left')

In [122]:
# Merge Annual Asset Growth into Monthly Panel

# 1. Prepare annual asset-growth characteristic

# fund_annual contains one observation per stock and
# fiscal/accounting year.
#
# asset_growth(t) =
#     Total Assets(t) / Total Assets(t-1) - 1
#
# For July formation in year T, we use:
#
#     FY(T-1) asset growth
#
# Therefore:
#
#     fiscal_year + 1 = formation_year

asset_growth_formation = fund_annual[
    [
        'symbol',
        'year',
        'asset_growth'
    ]
].copy()

asset_growth_formation = asset_growth_formation.rename(
    columns={
        'year': 'fiscal_year'
    }
)

asset_growth_formation['formation_year'] = (
    asset_growth_formation['fiscal_year'] + 1
)

asset_growth_formation = asset_growth_formation[
    [
        'symbol',
        'formation_year',
        'asset_growth'
    ]
].copy()


# 2. Remove any existing asset_growth from final

# final already received columns from fund_monthly earlier.
# If asset_growth exists there, we do NOT want to keep it,
# because that may be the old/monthly version.
#
# We want only the correctly constructed annual
# FY(t-1) investment characteristic.

asset_growth_cols = [
    col for col in final.columns
    if col.startswith('asset_growth')
]

if asset_growth_cols:
    final = final.drop(columns=asset_growth_cols)


# 3. Create portfolio formation year

# July 2020 -> formation year 2020
# August 2020 -> formation year 2020
# ...
# June 2021 -> formation year 2020
#
# July 2021 -> formation year 2021

final['formation_year'] = np.where(
    final['date'].dt.month >= 7,
    final['date'].dt.year,
    final['date'].dt.year - 1
)


# 4. Merge annual investment characteristic
final = final.merge(
    asset_growth_formation,
    on=['symbol', 'formation_year'],
    how='left'
)


# 5. Validation
print(
    "Asset growth non-null:",
    final['asset_growth'].notna().sum()
)

print(
    "Asset growth columns:",
    [
        col for col in final.columns
        if 'asset_growth' in col
    ]
)

Asset growth non-null: 26051
Asset growth columns: ['asset_growth']


In [125]:
final[['year_x', 'year_y']].head()

,year_x,year_y
0,2011,NaN
1,2011,NaN
2,2011,NaN
3,2011,NaN
4,2011,NaN


In [126]:
final.drop('year_y', axis=1, inplace=True)

In [129]:
final = final.rename(columns = {'year_x': 'year'})

In [130]:
print(
    final[
        [
            'date',
            'symbol',
            'year',
            'effective_year',
            'formation_year',
            'asset_growth'
        ]
    ]
    .dropna(subset=['asset_growth'])
    .head(20)
)

           date     symbol  year  effective_year  formation_year  asset_growth
7095 2013-07-31  TATAELXSI  2013            2013            2013      0.109272
7225 2013-07-31        ABB  2013            2013            2013      0.029277
7279 2013-08-31        ABB  2013            2013            2013      0.029277
7417 2013-08-31  TATAELXSI  2013            2013            2013      0.109272
7570 2013-09-30        ABB  2013            2013            2013      0.029277
7625 2013-09-30  TATAELXSI  2013            2013            2013      0.109272
7887 2013-10-31  TATAELXSI  2013            2013            2013      0.109272
7933 2013-10-31        ABB  2013            2013            2013      0.029277
8076 2013-11-30  TATAELXSI  2013            2013            2013      0.109272
8204 2013-11-30        ABB  2013            2013            2013      0.029277
8369 2013-12-31        ABB  2013            2013            2013      0.029277
8392 2013-12-31  TATAELXSI  2013            2013    

In [131]:
# ==========================================================
# Merge Book Value (FINAL STEP)
# ==========================================================

# Rename the duplicate column created by the fund_monthly merge if present
if 'book_value_x' in final.columns:
    final = final.drop(columns=[c for c in final.columns if c == 'book_value_y'])
    final = final.rename(columns={'book_value_x': 'book_value'})

print("Book value after merge:", final['book_value'].notna().sum())
print("Final shape:", final.shape)
print(final[['symbol', 'date', 'book_value']].dropna(subset=['book_value']).head(10))

Book value after merge: 31032
Final shape: (45664, 31)
         symbol       date  book_value
1205  TATAELXSI 2011-06-30       182.0
1330        ABB 2011-06-30      2534.0
1406  TATAELXSI 2011-07-31       182.0
1493        ABB 2011-07-31      2534.0
1698        ABB 2011-08-31      2534.0
1789  TATAELXSI 2011-08-31       182.0
1959  TATAELXSI 2011-09-30       182.0
2060        ABB 2011-09-30      2534.0
2088        ABB 2011-10-31      2534.0
2112  TATAELXSI 2011-10-31       182.0


In [132]:
final.head()

,date,"symbol""",series,prevclose,openprice,highprice,lowprice,lastprice,closeprice,averageprice,...,quarter,net_income,total_assets,source,book_value,profitability,index,Rf_monthly,formation_year,asset_growth
0,2011-01-31,UBL,EQ,443.65,435.55,484.0,425.2,482.4,474.00,458.36,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN
1,2011-01-31,BRITANNIA,EQ,368.35,371.9,382.0,365.0,380.95,378.30,375.88,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN
2,2011-01-31,MAGMA,EQ,60.65,59.9,60.85,58.0,60.0,60.05,59.74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN
3,2011-01-31,GABRIEL,EQ,45.25,45.0,45.0,41.8,42.6,42.70,42.65,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN
4,2011-01-31,PFC,EQ,254.45,247.2,251.0,240.55,245.85,245.20,245.36,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN


In [133]:
final.shape

(45664, 31)

In [134]:
final.isnull().sum()

date                       0
symbol"                    0
series                     0
prevclose                  0
openprice                  0
highprice                  0
lowprice                   0
lastprice                  0
closeprice                 0
averageprice               0
totaltradedquantity        0
turnoverinrs               0
no.oftrades             1142
deliverableqty             0
%dlyqttotradedqty          0
symbol                     0
return                   246
year                       0
month                      0
effective_year             0
turnover                   0
quarter                13903
net_income             14632
total_assets           14632
source                 13903
book_value             14632
profitability          14632
index                   3036
Rf_monthly              3036
formation_year             0
asset_growth           19613
dtype: int64

In [135]:
# =====================================================================
# 7.3 Market Cap
# =====================================================================

Market Cap = Price × Shares Outstanding

In [136]:
shares_history = {}

for sym in price_data['symbol'].unique():
    try:
        ticker = yf.Ticker(f"{sym}.NS")
        sh = ticker.get_shares_full(start="2011-01-01")
        if sh is not None and not sh.empty:
            sh = sh.resample("M").last().ffill()
            shares_history[sym] = sh
    except:
        shares_history[sym] = None

# Build a long-format DataFrame instead of a static map
shares_frames = []
for sym, sh in shares_history.items():
    if sh is not None:
        temp = sh.reset_index()
        temp.columns = ['date', 'shares_outstanding']
        temp['date'] = pd.to_datetime(temp['date']).dt.tz_localize(None)  # strip timezone
        temp['date'] = temp['date'] + pd.offsets.MonthEnd(0)
        temp['symbol'] = sym
        shares_frames.append(temp)

shares_df = pd.concat(shares_frames, ignore_index=True)

# Merge by both symbol AND date — not a static map
final = final.merge(shares_df, on=['symbol', 'date'], how='left')
final['market_cap'] = final['closeprice'] * final['shares_outstanding']
final['market_cap'] = pd.to_numeric(final['market_cap'], errors='coerce')

# Debug
print("Non-null shares:", final['shares_outstanding'].notna().sum())
print("Non-null market cap:", final['market_cap'].notna().sum())


Non-null shares: 25522
Non-null market cap: 25522


In [137]:
# ==========================================================
# SAVE MARKET CAP PANEL
# ==========================================================

market_cap_df = final[
    ['date', 'symbol', 'market_cap']
].copy()

market_cap_df.to_csv(
    PROCESSED_DIR / "market_cap_monthly.csv",
    index=False
)

print("Saved: market_cap_monthly.csv")

Saved: market_cap_monthly.csv


In [138]:
print("Total stocks:", len(shares_history))

print(
    "Stocks with historical shares:",
    sum(v is not None for v in shares_history.values())
)

print(
    "Stocks missing shares:",
    sum(v is None for v in shares_history.values())
)

print(
    "Rows with market cap:",
    final['market_cap'].notna().sum()
)

Total stocks: 246
Stocks with historical shares: 246
Stocks missing shares: 0
Rows with market cap: 25522


In [139]:
coverage = (
    final.groupby('date')['market_cap']
    .apply(lambda x: x.notna().mean())
)

print(coverage.describe())

count    188.000000
mean       0.553942
std        0.421009
min        0.000000
25%        0.042606
50%        0.861789
75%        0.906670
max        1.000000
Name: market_cap, dtype: float64


In [140]:
# =====================================================================
# 7.4 Book-to-Market
# =====================================================================

* Book Value = Shareholders' Equity (Net Worth)
* B/M = Book Value / Market Cap

In [141]:
final.head()

,date,"symbol""",series,prevclose,openprice,highprice,lowprice,lastprice,closeprice,averageprice,...,total_assets,source,book_value,profitability,index,Rf_monthly,formation_year,asset_growth,shares_outstanding,market_cap
0,2011-01-31,UBL,EQ,443.65,435.55,484.0,425.2,482.4,474.00,458.36,...,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN,NaN,NaN
1,2011-01-31,BRITANNIA,EQ,368.35,371.9,382.0,365.0,380.95,378.30,375.88,...,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN,NaN,NaN
2,2011-01-31,MAGMA,EQ,60.65,59.9,60.85,58.0,60.0,60.05,59.74,...,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN,NaN,NaN
3,2011-01-31,GABRIEL,EQ,45.25,45.0,45.0,41.8,42.6,42.70,42.65,...,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN,NaN,NaN
4,2011-01-31,PFC,EQ,254.45,247.2,251.0,240.55,245.85,245.20,245.36,...,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN,NaN,NaN


In [142]:
final.columns

Index(['date', 'symbol"', 'series', 'prevclose', 'openprice', 'highprice',
       'lowprice', 'lastprice', 'closeprice', 'averageprice',
       'totaltradedquantity', 'turnoverinrs', 'no.oftrades', 'deliverableqty',
       '%dlyqttotradedqty', 'symbol', 'return', 'year', 'month',
       'effective_year', 'turnover', 'quarter', 'net_income', 'total_assets',
       'source', 'book_value', 'profitability', 'index', 'Rf_monthly',
       'formation_year', 'asset_growth', 'shares_outstanding', 'market_cap'],
      dtype='object')

In [145]:
final[['effective_year', 'year']].isnull().sum()

effective_year    0
year              0
dtype: int64

In [146]:
len(final)

45664

In [147]:
# 1: December market cap
# dec_me -> dec market equity

dec_me = final[final['month'] == 12][
    ['symbol', 'year', 'market_cap']
].copy()

dec_me = dec_me.rename(
    columns={
        'year': 'dec_year',
        'market_cap': 'dec_market_cap'
    }
)


In [148]:
# 2: Previous fiscal-year book equity

# Book equity from fiscal year t-1.
# Take the latest available accounting observation
# for each stock within each fiscal year.

book_equity_annual = (
    final[
        final['book_value'].notna()
    ]
    .sort_values(['symbol', 'year', 'month'])
    .groupby(['symbol', 'year'], as_index=False)
    .last()
    [['symbol', 'year', 'book_value']]
)

book_equity_annual = book_equity_annual.rename(
    columns={
        'year': 'fiscal_year',
        'book_value': 'book_equity'
    }
)


In [149]:
# 3: Create formation year

# For July formation in year t:
#   Book Equity  -> FY t-1
#   December ME  -> December t-1
#
# Therefore:
#   formation_year = fiscal_year + 1

book_equity_annual['formation_year'] = (
    book_equity_annual['fiscal_year'] + 1
)

dec_me['formation_year'] = (
    dec_me['dec_year'] + 1
)


In [150]:
# 4: Merge book equity with December market cap

bm_formation = book_equity_annual.merge(
    dec_me[
        [
            'symbol',
            'formation_year',
            'dec_market_cap'
        ]
    ],
    on=['symbol', 'formation_year'],
    how='inner'
)


In [151]:
# 5: Calculate B/M

bm_formation['bm'] = (
    bm_formation['book_equity']
    / bm_formation['dec_market_cap']
)

# Keep only economically meaningful observations
bm_formation = bm_formation[
    (bm_formation['book_equity'] > 0) &
    (bm_formation['dec_market_cap'] > 0) &
    np.isfinite(bm_formation['bm']) &
    (bm_formation['bm'] > 0)
].copy()


In [152]:
bm_formation.head()

,symbol,fiscal_year,book_equity,formation_year,dec_market_cap,bm
2,AARTIIND,2017,1362.0,2018,9.354550e+10,1.455976e-08
3,AARTIIND,2018,1579.0,2019,1.175875e+11,1.342830e-08
4,AARTIIND,2019,2630.0,2020,1.446934e+11,1.817637e-08
5,AARTIIND,2020,2979.0,2021,2.151886e+11,1.384367e-08
6,AARTIIND,2021,3503.0,2022,3.683205e+11,9.510739e-09


In [153]:
final.columns

Index(['date', 'symbol"', 'series', 'prevclose', 'openprice', 'highprice',
       'lowprice', 'lastprice', 'closeprice', 'averageprice',
       'totaltradedquantity', 'turnoverinrs', 'no.oftrades', 'deliverableqty',
       '%dlyqttotradedqty', 'symbol', 'return', 'year', 'month',
       'effective_year', 'turnover', 'quarter', 'net_income', 'total_assets',
       'source', 'book_value', 'profitability', 'index', 'Rf_monthly',
       'formation_year', 'asset_growth', 'shares_outstanding', 'market_cap'],
      dtype='object')

In [154]:
final[['date', 'prevclose', 'openprice', 'highprice', 'lowprice', 'lastprice', 'closeprice', 'symbol', 'return', 'year', 
       'month', 'effective_year']].head()

,date,prevclose,openprice,highprice,lowprice,lastprice,closeprice,symbol,return,year,month,effective_year
0,2011-01-31,443.65,435.55,484.0,425.2,482.4,474.00,UBL,NaN,2011,1,2010
1,2011-01-31,368.35,371.9,382.0,365.0,380.95,378.30,BRITANNIA,NaN,2011,1,2010
2,2011-01-31,60.65,59.9,60.85,58.0,60.0,60.05,POONAWALLA,NaN,2011,1,2010
3,2011-01-31,45.25,45.0,45.0,41.8,42.6,42.70,GABRIEL,NaN,2011,1,2010
4,2011-01-31,254.45,247.2,251.0,240.55,245.85,245.20,PFC,NaN,2011,1,2010


In [155]:
# 6: Create formation year and merge B/M

# FF portfolio formation year:
#
# July 2020 -> June 2021 = formation year 2020
# July 2021 -> June 2022 = formation year 2021
#
# Therefore:
# July-Dec -> current calendar year
# Jan-Jun  -> previous calendar year

final['formation_year'] = np.where(
    final['month'] >= 7,
    final['year'],
    final['year'] - 1
)

# Merge the formation-year B/M back to the monthly panel
final = final.merge(
    bm_formation[
        [
            'symbol',
            'formation_year',
            'bm'
        ]
    ],
    on=['symbol', 'formation_year'],
    how='left'
)

print("B/M non-null:", final['bm'].notna().sum())

print(
    final[
        [
            'date',
            'symbol',
            'year',
            'effective_year',
            'formation_year',
            'bm'
        ]
    ]
    .dropna(subset=['bm'])
    .head(20)
)

B/M non-null: 21907
            date      symbol  year  effective_year  formation_year  \
12998 2015-07-31         ACC  2015            2015            2015   
13075 2015-08-31         ACC  2015            2015            2015   
13526 2015-09-30         ACC  2015            2015            2015   
13602 2015-10-31         ACC  2015            2015            2015   
13866 2015-11-30         ACC  2015            2015            2015   
14109 2015-12-31         ACC  2015            2015            2015   
14432 2016-01-31         ACC  2016            2015            2015   
14642 2016-02-29         ACC  2016            2015            2015   
14976 2016-03-31         ACC  2016            2015            2015   
15254 2016-04-30         ACC  2016            2015            2015   
15365 2016-05-31         ACC  2016            2015            2015   
15627 2016-06-30         ACC  2016            2015            2015   
15756 2016-07-31     SIEMENS  2016            2016            2016   


In [156]:
print("Final shape before filters:", final.shape)
print("Market cap non-null:", final['market_cap'].notna().sum())
print("Book value non-null:", final['book_value'].notna().sum())
print("BM non-null:", final['bm'].notna().sum())

Final shape before filters: (45664, 34)
Market cap non-null: 25522
Book value non-null: 31032
BM non-null: 21907


In [157]:
final = final[
    final['market_cap'] > 0
].copy()

In [158]:
final.columns

Index(['date', 'symbol"', 'series', 'prevclose', 'openprice', 'highprice',
       'lowprice', 'lastprice', 'closeprice', 'averageprice',
       'totaltradedquantity', 'turnoverinrs', 'no.oftrades', 'deliverableqty',
       '%dlyqttotradedqty', 'symbol', 'return', 'year', 'month',
       'effective_year', 'turnover', 'quarter', 'net_income', 'total_assets',
       'source', 'book_value', 'profitability', 'index', 'Rf_monthly',
       'formation_year', 'asset_growth', 'shares_outstanding', 'market_cap',
       'bm'],
      dtype='object')

In [159]:
# ==========================================================
# SAVE CHARACTERISTICS PANEL
# ==========================================================

characteristics = final[
    [
        'date',
        'symbol',
        'market_cap',
        'bm',
        'profitability',
        'asset_growth'
    ]
].copy()

characteristics.to_csv(
    PROCESSED_DIR / "characteristics_monthly.csv",
    index=False
)

print("Saved: characteristics_monthly.csv")

Saved: characteristics_monthly.csv


In [160]:
final.shape

(25522, 34)

In [161]:
print(final.columns)

Index(['date', 'symbol"', 'series', 'prevclose', 'openprice', 'highprice',
       'lowprice', 'lastprice', 'closeprice', 'averageprice',
       'totaltradedquantity', 'turnoverinrs', 'no.oftrades', 'deliverableqty',
       '%dlyqttotradedqty', 'symbol', 'return', 'year', 'month',
       'effective_year', 'turnover', 'quarter', 'net_income', 'total_assets',
       'source', 'book_value', 'profitability', 'index', 'Rf_monthly',
       'formation_year', 'asset_growth', 'shares_outstanding', 'market_cap',
       'bm'],
      dtype='object')


In [162]:
# ==========================================================
# FINAL COLUMN CLEANUP
# ==========================================================

# Remove corrupted NSE column
final = final.drop(columns=['symbol"'], errors='ignore')

# Remove duplicate accounting year
final = final.drop(columns=['year_y'], errors='ignore')

# Rename market year
final = final.rename(columns={'year_x': 'year'})

# Optional: reorder important columns
important_cols = [
    'date',
    'symbol',
    'year',
    'month',
    'return',
    'Rf_monthly',
    'market_cap',
    'bm',
    'profitability',
    'asset_growth'
]

remaining_cols = [
    c for c in final.columns
    if c not in important_cols
]

final = final[important_cols + remaining_cols]

In [163]:
# ==========================================================
# SAVE ELIGIBLE STOCK UNIVERSE
# ==========================================================

eligible = final[
    ['date', 'symbol']
].drop_duplicates()

eligible.to_csv(
    PROCESSED_DIR / "eligible_universe.csv",
    index=False
)

print("Saved: eligible_universe.csv")

Saved: eligible_universe.csv


In [164]:
# ==========================================================
# SAVE MASTER MONTHLY PANEL
# ==========================================================

PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

final.to_csv(
    PROCESSED_DIR / "final_panel.csv",
    index=False
)

print("Saved: final_panel.csv")

Saved: final_panel.csv


---

## Phase 1 Summary — Data Ingestion & Cleaning ✓

### What was achieved

| Dataset | Coverage | Stored at |
|---|---|---|
| Price + volume (NSE EQ series) | 2011–present, ~470 stocks, monthly | `data/raw/*.csv` |
| Fundamentals: net income, total assets, book value | 2011–present, 476 stocks, annual (Screener) | `data/fin_data_processed/final_fin_data.csv` |
| Risk-free rate (91-day T-bill, FRED) | 2011–present, monthly compounded | `data/rf_india.csv` |

**Factor inputs derived:**
- `profitability` = Net Income / Total Assets → feeds **RMW**
- `asset_growth` = % change in Total Assets → feeds **CMA**
- `book_value` (lagged 3 months, annual → monthly expanded) → feeds **HML**
- `market_cap` = Price × Shares Outstanding → feeds **SMB** and **HML**
- `bm` = Book Value / Market Cap → feeds **HML**
- `return` and `Rf_monthly` → feed **MKT**

All fundamentals are lagged 3 months to prevent lookahead bias. Stocks with fewer than 70% monthly observations or median volume below 100k shares were dropped.

---

### How this feeds Phase 2 — Factor Construction

In Phase 2 we use `final` (the merged monthly panel) to construct all five Fama–French factors:

- **MKT** — value-weighted market return minus `Rf_monthly`, computed across all stocks each month
- **SMB** — monthly return spread between bottom and top tercile of `market_cap`, size-sorted portfolios
- **HML** — monthly return spread between top and bottom tercile of `bm`, annually rebalanced each July
- **RMW** — monthly return spread between top and bottom tercile of `profitability`
- **CMA** — monthly return spread between bottom and top tercile of `asset_growth` (low investment minus high)

Portfolio sorts follow the standard FF methodology: independent 2×3 sorts on size and each characteristic, rebalanced annually in July using prior fiscal year-end fundamentals (already lagged in Phase 1).